## SQL Analysis

This notebook replicates and extends the EDA findings using pure SQL queries.

All queries run on a SQLite database ('bank.db') loaded from the clean dataset.

In [2]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [71]:
pd.set_option('display.max_rows', None) 
pd.set_option('display.max_columns', None)

In [3]:
# Load the clean dataset
df= pd.read_csv('../data/processed_data/bank_clean.csv')

# Create the database
connection= sqlite3.connect('bank.db')

# Load dataframe into SQL table
df.to_sql('customer', connection, if_exists='replace', index=False)
print(f"Database created - {len(df)} rows loaded into 'customer' table")


Database created - 41172 rows loaded into 'customer' table


In [4]:
# Look at the entire table
query= """
SELECT *
FROM customer
"""
pd.read_sql(query, connection)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,previously_contacted,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,0,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41167,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41168,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41169,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41170,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,0,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


## Table of Contents

1. Customer Demographics Analysis
2. Financial Profile Analysis
3. Customer History & Behavioral Analysis
4. Campaign Strategy Analysis
5. Economic Conditions Analysis
6. Temporal & Seasonal Analysis
7. Which segments are strategically valuable? (most important section)

## 1. Customer Demographics Analysis
This section focuses on understanding which customer profiles are more likely to subscribe to the bank product.

The goal is to identify the most responsive customer segments based on demographic characteristics.

## 1.1 Conversion Rate by Job Category

Which professions show the highest convertion rates?

In [7]:
# Conversion rate by job
query="""
SELECT 
    job, COUNT(*) as total_per_job,
    SUM(CASE WHEN y= 'yes' THEN 1 ELSE 0 END) as subscriptions,
    ROUND(AVG(CASE WHEN y= 'yes' THEN 1 ELSE 0 END) *100,2) as conversion_rate
FROM customer
GROUP BY job
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,job,total_per_job,subscriptions,conversion_rate
0,student,875,275,31.43
1,retired,1718,434,25.26
2,unemployed,1014,144,14.20
3,admin.,10748,1388,12.91
4,management,2923,328,11.22
5,technician,6739,730,10.83
6,self-employed,1421,149,10.49
7,housemaid,1060,106,10.00
8,entrepreneur,1456,124,8.52
9,services,3967,323,8.14


### Key Findings

Students and retired clients show significantly higher conversion rates 
(31.4% and 25.3% respectively).

Students may be more receptive to financial products 
as they begin building savings, while retired clients typically have more 
disposable capital to invest in term deposits.

Blue-collar workers show the lowest conversion rate (6.9%), 
likely reflecting lower disposable income and less familiarity 
with investment products.

**Business recommendation:** future campaigns should prioritize 
student and retired segments for higher ROI.

## 1.2 Job & Marital Status Analysis
What job and marital status combinations convert better?

**Objective**: understand whether marital status affects conversion behavior within specific professions.

In [10]:
query="""
SELECT 
   job, marital,
   COUNT(*) as total_per_group,
   SUM(CASE WHEN y= 'yes' THEN 1 ELSE 0 END) as subscriptions,
   ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
GROUP BY job, marital
HAVING COUNT(*)>50
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,job,marital,total_per_group,subscriptions,conversion_rate
0,student,single,824,264,32.04
1,retired,divorced,348,92,26.44
2,retired,married,1278,330,25.82
3,unemployed,single,251,48,19.12
4,admin.,single,3949,581,14.71
5,self-employed,single,379,51,13.46
6,unemployed,married,639,86,13.46
7,housemaid,single,119,16,13.45
8,retired,single,92,12,13.04
9,management,single,501,63,12.57


## Key findings
1. **Single students**, show the highest conversion rate in the dataset.This suggests that younger customers may be more receptive to the campaign or more interested in the offered financial product, maybe beacuse they have less expenses.

2. Retired customers also show very high conversion rates across all marital statuses.

3. Single unemployed customers have a relatively high conversion rate, this may indicate increased interest in financial security or savings products.

4. **Single vs married**: across several professions, single customers tend to have higher conversion rates than married customers. **This pattern suggests marital status may influence responsiveness to marketing campaigns**.

5. Blue-collare and services have lower conversion rates, these segments may require different communication strategies or may be less interested in the product offered.

## 2. Financial Profile Analysis
How does overall debt exposure influence conversion?

## 2.1 Conversion rate by housing and loan

In [14]:
query="""
SELECT
    housing, loan,
    COUNT(*) as total,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
GROUP BY housing, loan
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,housing,loan,total,conversion_rate
0,yes,no,18867,11.69
1,yes,yes,3691,11.08
2,no,no,16057,10.90
3,no,yes,2557,10.72


## 2.2 Financial Debt Exposure & Interest Rate Environment
Do customers with housing and personal loans behave differently during periods of high interest rates?

Following the segmentation in Section 5, I will use the two defined economic period groups here.

In [16]:
query="""
SELECT
    housing, loan,
    CASE
    WHEN euribor3m < 2.0 THEN 'Low-Rate'
    ELSE 'High-Rate'
    END as economic_period,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
GROUP BY housing, loan, economic_period
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,housing,loan,economic_period,conversion_rate
0,no,no,Low-Rate,24.85
1,no,yes,Low-Rate,24.68
2,yes,no,Low-Rate,24.59
3,yes,yes,Low-Rate,22.18
4,yes,yes,High-Rate,5.15
5,no,no,High-Rate,4.97
6,yes,no,High-Rate,4.68
7,no,yes,High-Rate,4.66


## Key Insights
- Conversion rates drop sharply during high-rate economic periods across all financial profiles.
- During low-rate environments, conversion rates remain consistently high regardless of loan status - customers may feel financially comfortable enough to engage with banking products when interest rates are low.
- Customers with both loans perform slightly worse.

**Economic Conditions Matter More Than Loan Status**: the difference between low-rate and high-rate environments is much larger than the difference between financial profiles - macroeconomic conditions are a stronger driver of customer behavior than debt exposure alone.

**Business recommendation**: adapt campaign timing to economic conditions and better identify financially sensitive customer segments to personalize communication strategies.

## 3. Customer History & Behavioral Analysis

This section investigates whether previous interactions with customers influence future conversions.

The goal is to understand how customer history impacts campaign success.

## 3.1 Job & Previous Campaign Outcome
How does previous campaign success affect different job categories?

In [20]:
query="""
SELECT
    job, poutcome,
    COUNT(*) as total_per_group,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0, 2) as conversion_rate
FROM customer
WHERE poutcome != 'nonexistent'
GROUP BY job, poutcome
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,job,poutcome,total_per_group,conversion_rate
0,retired,success,158,74.05
1,student,success,118,71.19
2,housemaid,success,38,71.05
3,management,success,95,69.47
4,unemployed,success,62,66.13
5,technician,success,211,65.88
6,admin.,success,447,65.32
7,services,success,70,60.00
8,self-employed,success,30,53.33
9,blue-collar,success,119,49.58


## Key insights
Customers with a previous successful campaign outcome show extremely high conversion rates across nearly all job categories.

**Previous campaign success** is the **strongest predictor** of conversion.

Customers with previous failed campaign outcomes convert at much lower rates across all professions - unsuccessful previous interactions strongly reduce future responsiveness. But certain customer groups remain relatively resilient - they maintain moderate conversion rates despite these previous campaign failures (retire|failure -> 27.39% and student|failure -> 26.38%).

**Business recommendation**
- Prioritize customers with positive campaign histories,
- Optimize retargeting strategies,
- Reduce effort on historically unresponsive segments.

## 4. Campaign Strategy Analysis

This section evaluates how campaign execution influences conversion performance.

The focus is on contact strategy, communication channels, and campaign intensity.

## 4.1 Contact Type Analysis (Cellular vs Telephone)
Which communication channel performs better?

In [24]:
query="""
SELECT
    contact, COUNT(*) as total_contacts,
    SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as absolute_subs,
    ROUND(AVG(CASE WHEN Y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
GROUP BY contact
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,contact,total_contacts,absolute_subs,conversion_rate
0,cellular,26132,3852,14.74
1,telephone,15040,787,5.23


## What's the best communication channel?
Cellular contact significantly outperforms telephone, 
confirming that mobile outreach is the more effective channel 
for this specific type of campaign.

## 4.2 Optimal Number of Contacts
How does the number of campaign contacts affect conversion rates?

**Objective**: determine whether excessive contact attempts reduce campaign effectiveness.

In [27]:
query="""
SELECT
    campaign, Count(*) as total,
    SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as subscriptions,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0, 2) as conversion_rate
FROM customer
GROUP BY campaign
HAVING total > 50
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,campaign,total,subscriptions,conversion_rate
0,1,17634,2299,13.04
1,2,10567,1211,11.46
2,3,5339,574,10.75
3,4,2649,249,9.40
4,6,979,75,7.66
5,5,1599,120,7.50
6,17,58,4,6.90
7,11,177,12,6.78
8,7,629,38,6.04
9,9,283,17,6.01


## Key insights - Optimal Number

Conversion rate drops consistently as the number of contacts increases. 
Clients contacted only once convert at 13%, dropping to 9.4% by the fourth contact 
and plateauing around 6-7% beyond that.

**Business recommendation:** beyond 3-4 contacts, additional calls produce 
strongly diminishing returns. Campaign resources should be reallocated 
toward fresh leads rather than persistent follow-up.

**Keep in mind this does not contradict the poutcome finding** — clients who subscribed 
in a *previous* campaign are still high-value targets. The distinction is 
between persistent follow-up within the same campaign (ineffective) 
and re-engaging previously successful clients in future campaigns (highly effective).

## 4.3 Campaign Intensity & Previous Outcome
Does contact frequency affect customers differently depending on previous campaign outcomes?

**Objective**: analyze whether highly contacted customers continue to respond positively after previous successful campaigns.

In [30]:
query="""
SELECT
    campaign, poutcome,
    COUNT(*) as total,
    SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as subscriptions,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
WHERE poutcome != 'nonexistent'
GROUP BY campaign, poutcome
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,campaign,poutcome,total,subscriptions,conversion_rate
0,3,success,144,99,68.75
1,1,success,724,487,67.27
2,2,success,400,261,65.25
3,6,success,17,10,58.82
4,4,success,48,26,54.17
5,5,success,26,9,34.62
6,10,failure,6,2,33.33
7,7,success,9,2,22.22
8,4,failure,224,35,15.63
9,3,failure,425,65,15.29


Customers with previous successful campaign outcomes maintain very high conversion rates during the first 3 contacts.

Conversion rates begin to decline after multiple contact attempts, especially after 4–5 campaigns.

Customers with previous failed outcomes consistently underperform across nearly all campaign intensities.

## 5. Economic Conditions Analysis

This section explores the impact of macroeconomic conditions on customer behavior.

The **goal** is to understand whether economic environments influence subscription decisions and to measure this relationship.

## 5.1 Conversion Rate by Euribor Level
Do customers convert differently under low vs high interest rate environments?

**Important**: in the EDA, the scatter plot revealed two distinct clusters with no data points 
between euribor 1.8 and 4.0 — confirming the dataset spans two separate 
economic periods. 
I use 2.0 as a natural cutoff to split clients into low-rate and high-rate periods.

In [34]:
query="""
SELECT
    CASE
    WHEN euribor3m < 2.0 THEN 'Low-Rate'
    ELSE 'High-rate' 
    END as economic_period,
    COUNT(*) as total,
    SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as subscriptions,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
GROUP BY economic_period
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)
    

,economic_period,total,subscriptions,conversion_rate
0,Low-Rate,13492,3300,24.46
1,High-rate,27680,1339,4.84


## 5.2 Job Category & Euribor Level
How do economic conditions affect different professional categories?

**Objective**: understand whether certain customer segments are more sensitive to interest rate changes.

In [36]:
query="""
SELECT
    job,
    CASE
    WHEN euribor3m < 2.0 THEN 'Low-Rate'
    ELSE 'High-Rate'
    END as economic_period,
    SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as subscriptions,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
GROUP BY  job, economic_period
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,job,economic_period,subscriptions,conversion_rate
0,retired,Low-Rate,387,41.93
1,student,Low-Rate,261,38.27
2,unemployed,Low-Rate,120,34.09
3,housemaid,Low-Rate,79,33.47
4,admin.,Low-Rate,1039,28.28
5,technician,Low-Rate,501,26.81
6,management,Low-Rate,220,24.20
7,self-employed,Low-Rate,101,23.54
8,entrepreneur,Low-Rate,66,16.92
9,services,Low-Rate,191,15.30


## Key findings
- The numbers confirm what the EDA suggested visually — the difference is really big, during low-rate periods, nearly 1 in 4 clients subscribed (24.5%), compared to less than 1 in 20 during high-rate periods (4.8%). This 5x difference makes the macroeconomic context the one of the strongest 
predictor of campaign success in the entire dataset.

- The gap between economic periods is consistent across all job categories — every segment converts significantly better during low-rate periods. However, the magnitude of the effect varies considerably:

1. retired and student clients show the highest conversion rates in low-rate 
periods (41.9% and 38.3% respectively), but collapse to around 6-7% 
during high rates — a drop of over 30 percentage points. 
These segments (so eve the best-performing segments individually in terms of conversion) are highly sensitive to economic conditions.

2. blue-collar and services workers show the smallest absolute drop, 
but only because their baseline was already low to begin with.

**Business recommendation:** campaign timing matters as much as targeting. 
Even the best customer segments are unlikely to convert 
during high-rate periods. The bank should concentrate its highest-volume 
outreach efforts during favorable macroeconomic windows, 
and maintain only minimal contact during high-rate periods 
to preserve client relationships without wasting resources.

## 6. Temporal & Seasonal Analysis

This section investigates how timing influences campaign performance.

The goal is to identify seasonal patterns and optimal campaign periods.

## 6.1 Job Category & Month Analysis
Do some customer segments perform better during specific months?

**Objective**: analyze seasonal conversion patterns across customer professions.

In [73]:
query="""
SELECT
    job, month,
    COUNT(*) as total_per_group,
    SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as subscriptions,
    ROUND(AVG(CASE WHEN y='yes' THEN 1 ELSE 0 END)*100.0,2) as conversion_rate
FROM customer
GROUP BY job, month
HAVING total_per_group > 50
ORDER BY conversion_rate DESC
"""
pd.read_sql(query, connection)

,job,month,total_per_group,subscriptions,conversion_rate
0,admin.,mar,171,89,52.05
1,management,oct,51,26,50.98
2,student,oct,53,27,50.94
3,admin.,sep,174,87,50.00
4,technician,oct,116,58,50.00
5,retired,mar,64,31,48.44
6,technician,mar,102,49,48.04
7,technician,sep,92,44,47.83
8,admin.,oct,205,91,44.39
9,admin.,dec,52,23,44.23


## Key Insights
-Some job categories (admin., management, student, technician) achieved exceptionally high conversion rates during specific months (March, September, October).

This suggests that campaign timing has a strong impact on customer responsiveness.

-Summer Months Have High Volume but Lower Efficiency.
May, June, July, and August contain much larger customer volumes but lower conversion rates.

This means that large-scale campaigns may reduce targeting quality and overall efficiency.

-Blue-collar customers tend to have lower conversion rates across most months.

This segment may require different marketing strategies or may be less interested in the product.

-Multiple professions show very high performance in October, this may indicate particularly effective campaign timing or favorable market conditions.

**Business Interpretation**: the analysis indicates that both customer profession and campaign timing significantly influence conversion behavior and shows how high-volume campaigns tend to reduce efficiency.

## 7. Which segments are strategically valuable?
We must identify the most profitable segments by looking beyond conversion rates and considering other factors like **volume** and **stability** as well.

## 7.1 Job Volume Analysis
Students and retired clients have higher conversion rates (as found in the first section), but that doesn’t necessarily mean they are the best segments to target.

It's important to look at the groups with the highest absolute number of subscriptions

In [44]:
query="""
SELECT
    job,
    SUM(CASE WHEN y='yes' THEN 1 ELSE 0 END) as total_subscriptions
FROM customer
GROUP BY job
ORDER BY total_subscriptions DESC
"""
pd.read_sql(query, connection)

,job,total_subscriptions
0,admin.,1388
1,technician,730
2,blue-collar,638
3,retired,434
4,management,328
5,services,323
6,student,275
7,self-employed,149
8,unemployed,144
9,entrepreneur,124


### Volume vs Conversion Rate

Looking at absolute subscription volumes alongside conversion rates 
reveals an important nuance for campaign prioritization.

Admin. and technician are the most important segments (in terms of pure volume) generating the highest absolute volume 
(1,388 and 730 subscriptions respectively), but simply because they are 
the most contacted segments, not because they convert particularly well 
(12.9% and 10.8%).

The most interesting segments are retired and students. Despite being 
contacted far less than admin or blue-collar workers, they generate 
434 and 275 subscriptions at conversion rates of 25% and 31% respectively. 
They are underrepresented in the campaign relative to their efficiency.

**Business recommendation:** the bank is over-investing in high-volume, 
low-conversion segments like blue-collar (638 subscriptions at 6.9%) 
while under-investing in high-efficiency segments like students and retired. 
Reallocating even a fraction of blue-collar contacts toward retired 
and student profiles (especially during low-rate periods)
would significantly improve overall campaign ROI. 
It's important to acknowledge the importance of 'admin.' and 'technician' groups that contribute the most financially, without considering acquisition costs or the actual marketing spend for a larger customer base (in a business 'average-performing' segments are often preferred over small, 'exceptional' ones).

## 7.2 Stability Analysis

To measure stability, we split clients into two economic periods 
using the euribor threshold identified earlier (< 2.0 = low rate, >= 2.0 = high rate) 
and calculated the conversion rate for each job in both contexts.

We chose economic context over monthly breakdown as the stability lens 
because (as shown in Section 3) euribor is a really strong predictor 
in the dataset. A segment that holds up across economic periods 
is genuinely reliable, not just seasonally lucky.

`rate_spread` measures the drop in conversion rate from low to high rate periods. The smaller the spread, the more stable the segment.

In [48]:
query="""
SELECT
    job,
    ROUND(AVG(CASE WHEN euribor3m < 2.0 AND y='yes' THEN 1 ELSE 0 END)*100.0, 2) as conv_low_rate,
    ROUND(AVG(CASE WHEN euribor3m > 2.0 AND y='yes' THEN 1 ELSE 0 END)*100.0, 2) as conv_high_rate,
    ROUND(AVG(CASE WHEN euribor3m < 2.0 AND y='yes' THEN 1 ELSE 0 END)*100.0 - AVG(CASE WHEN euribor3m > 2.0 AND y='yes' THEN 1 ELSE 0 END)*100.0, 2)
       as rate_spread
FROM customer
GROUP BY job
ORDER BY rate_spread ASC
"""
pd.read_sql(query, connection)

,job,conv_low_rate,conv_high_rate,rate_spread
0,blue-collar,3.62,3.28,0.35
1,entrepreneur,4.53,3.98,0.55
2,services,4.81,3.33,1.49
3,self-employed,7.11,3.38,3.73
4,management,7.53,3.69,3.83
5,technician,7.43,3.40,4.04
6,housemaid,7.45,2.55,4.91
7,admin.,9.67,3.25,6.42
8,unemployed,11.83,2.37,9.47
9,retired,22.53,2.74,19.79


## Key Insights — Stability vs Performance

The results reveal a clear trade-off between performance and stability.

**Most stable segments:** blue-collar and entrepreneur show the smallest spread 
(0.35 and 0.55 respectively), they convert consistently across both economic periods. 
However, their absolute conversion rates are the lowest in the dataset (3-4%). 
They are reliable but not profitable.

**Most volatile segments:** student and retired show the largest spreads 
(28.2 and 19.8 points), they are exceptional during low-rate periods 
(29.8% and 22.5%) but nearly disappear during high rates (1.6% and 2.7%). 
They are high-reward but highly context-dependent.

**The sweet spot:** **admin and technician** sit in the middle, 
moderate spread (6.4 and 4.0) with decent low-rate conversion (9.7% and 7.4%). 
They represent the **most balanced segments** for consistent campaign targeting 
regardless of economic conditions.

**Business recommendation:** a robust campaign strategy should combine 
two approaches:
1. during favorable economic windows, concentrate resources 
on retired and student segments for maximum ROI.
2. during high-rate periods, 
shift focus to admin and technician profiles — lower returns but more predictable, 
and large enough in volume.

## 7.3 Risk - Opportunity Matrix
Single-variable analysis tells who converts best in isolation. 
But, as seen above, conversion rate alone is not enough — 
a segment that converts for example at 30% but has only 50 clients is less actionable 
than one converting at 15% with 1000 clients.

Now I want to classify each job segment across the two analyzed dimensions simultaneously:
- **Conversion rate**
- **Volume**
I'll use the dataset averages as thresholds to split segments into four quadrants:

| Segment Type | Meaning |
|---|---|
| Top Priority | High conversion + high volume |
| Niche Opportunity | High conversion + low volume |
| Optimization Needed | Low conversion + high volume |
| Low Priority | Low conversion + low volume |

In this way cutomers are calssified from top priority to low priority.

**Note**: the averages are calculated within the query itself (using CTE).

In [62]:
query = """
WITH segment_type as (
    SELECT 
        job,
        COUNT(*) as total,
        ROUND(AVG(CASE WHEN y = 'yes' THEN 1.0 ELSE 0 END) * 100, 2) as conversion_rate
    FROM customer
    GROUP BY job
),
average as (
    SELECT 
        AVG(total) as avg_volume,
        AVG(conversion_rate) as avg_conversion
    FROM segment_type
)
SELECT 
    s.job,
    s.total,
    s.conversion_rate,
    CASE 
        WHEN s.total >= a.avg_volume AND s.conversion_rate >= a.avg_conversion 
            THEN 'Top Priority'
        WHEN s.total < a.avg_volume AND s.conversion_rate >= a.avg_conversion 
            THEN 'Niche Opportunity'
        WHEN s.total >= a.avg_volume AND s.conversion_rate < a.avg_conversion 
            THEN 'Optimization Needed'
        ELSE 'Low Priority'
    END as segment_classification
FROM segment_type s, average a
ORDER BY conversion_rate DESC
"""

pd.read_sql(query, connection)

,job,total,conversion_rate,segment_classification
0,student,875,31.43,Niche Opportunity
1,retired,1718,25.26,Niche Opportunity
2,unemployed,1014,14.20,Niche Opportunity
3,admin.,10748,12.91,Optimization Needed
4,management,2923,11.22,Low Priority
5,technician,6739,10.83,Optimization Needed
6,self-employed,1421,10.49,Low Priority
7,housemaid,1060,10.00,Low Priority
8,entrepreneur,1456,8.52,Low Priority
9,services,3967,8.14,Optimization Needed


**Business recommendation**

No segment qualifies as Top Priority.

- The three **highest-converting segments (student, retired, unemployed)**
al fall into Niche Opportunity (high efficiency but low volume). 
These are underleveraged -> **increasing outreach toward them
(especially during low-rate periods) would improve overall ROI 
without proportional cost increases**.

- The **largest segments by volume** (admin, technician, blue-collar, services) 
are all classified as Optimization Needed.
They absorb the majority of 
campaign resources but convert below average. 
**Reducing contact frequency on these segments and reallocating toward 
niche opportunities would be the highest impact operational change** 
the bank could make.